# 02 — Credit Score Classification

Dataset: **Credit Score Classification** (Kaggle) — 100k dòng, 28 feature, target 3 lớp (Poor/Standard/Good).

**Pipeline:** Load → Clean (regex + parse) → EDA → FE → Split-by-Customer → Train (LogReg / LightGBM / XGBoost) → Evaluate → Convert Prob → FICO Score → SHAP → Export.

**Ghi chú quan trọng:**
- Dataset là **panel data** — mỗi `Customer_ID` xuất hiện nhiều tháng → phải split train/test theo `Customer_ID` để tránh leakage.
- Dataset nổi tiếng "messy" — nhiều cột số bị dính ký tự rác, phải clean bằng regex.
- Sau khi có model classification, ta **convert probability → FICO score 300–850** bằng expected ordinal, rồi map sang 5 nhóm FICO chuẩn.

**Output:** `model.pkl`, `features.json` → copy vào `models/` để Streamlit dùng.

## 0. Setup

In [ ]:
# Chỉ chạy trên Colab — local đã cài qua requirements.txt
# !pip install -q lightgbm xgboost shap

In [ ]:
import json, re, warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (LabelEncoder, OneHotEncoder,
                                    OrdinalEncoder, StandardScaler)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)

RANDOM_STATE = 42

### Mount Google Drive (Colab)

In [ ]:
# ===== Colab: mount Drive =====
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/Công nghệ dịch vụ tài chính')
DATA_DIR = PROJECT_DIR / 'data' / 'Credit_score_classification'
OUT_DIR = PROJECT_DIR / 'models'

# ===== Local (bỏ comment nếu chạy local, comment 2 cell ở trên) =====
# PROJECT_DIR = Path('..')
# DATA_DIR = PROJECT_DIR / 'data' / 'Credit_score_classification'
# OUT_DIR = PROJECT_DIR / 'models'

OUT_DIR.mkdir(exist_ok=True, parents=True)
print('Data dir:', DATA_DIR)
print('Out dir :', OUT_DIR)

## 1. Load raw

In [ ]:
df = pd.read_csv(DATA_DIR / 'train.csv', low_memory=False)
print('Shape:', df.shape)
df.head()

In [ ]:
print('=== TARGET DISTRIBUTION ===')
print(df['Credit_Score'].value_counts())
print()
print('=== DTYPES ===')
print(df.dtypes)
print()
print('=== MISSING ===')
mis = df.isna().sum()
print(mis[mis > 0].sort_values(ascending=False))

## 2. Data Cleaning

Nhiều cột số bị dính ký tự rác (`_`, `__-333333__`, ...) → parse thành `float`. `Credit_History_Age` dạng chuỗi `"22 Years and 1 Months"` → tổng số tháng.

In [ ]:
def to_num(s):
    """Bỏ mọi ký tự không phải số/dấu chấm/dấu trừ, parse float."""
    if pd.isna(s):
        return np.nan
    cleaned = re.sub(r'[^0-9.\-]', '', str(s))
    try:
        return float(cleaned) if cleaned not in ('', '-', '.') else np.nan
    except ValueError:
        return np.nan

num_like_cols = [
    'Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment',
    'Changed_Credit_Limit', 'Outstanding_Debt', 'Amount_invested_monthly',
    'Monthly_Balance'
]
for c in num_like_cols:
    df[c] = df[c].apply(to_num)

def parse_history(s):
    if pd.isna(s):
        return np.nan
    m = re.match(r'(\d+)\s*Years?\s*(?:and\s*(\d+)\s*Months?)?', str(s))
    if m:
        y = int(m.group(1))
        mo = int(m.group(2)) if m.group(2) else 0
        return y * 12 + mo
    return np.nan

df['Credit_History_Months'] = df['Credit_History_Age'].apply(parse_history)

# Bảng chọn categorical: '_' là placeholder của missing
df['Credit_Mix'] = df['Credit_Mix'].replace('_', np.nan)
df['Payment_Behaviour'] = df['Payment_Behaviour'].replace('!@9#%8', np.nan)
df['Occupation'] = df['Occupation'].replace('_______', np.nan)

print('Sau khi clean:')
print(df[num_like_cols + ['Credit_History_Months']].dtypes)

In [ ]:
# Xử lý outliers vô lý — clip theo domain
clip_rules = {
    'Age': (14, 100),
    'Num_Bank_Accounts': (0, 20),
    'Num_Credit_Card': (0, 20),
    'Interest_Rate': (0, 40),
    'Num_of_Loan': (0, 20),
    'Num_of_Delayed_Payment': (0, 50),
    'Num_Credit_Inquiries': (0, 30),
}
for col, (lo, hi) in clip_rules.items():
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    df[col] = df[col].clip(lo, hi)
    print(f'{col}: clipped {n_out} outliers → [{lo}, {hi}]')

In [ ]:
# Missing values sau khi clean
mis = df.isna().sum()
print(mis[mis > 0].sort_values(ascending=False))

## 3. Feature Engineering

- `Type_of_Loan` là multi-label (comma-separated) → tách thành số lượng loan types + top-3 loại phổ biến nhất dưới dạng 0/1.
- `Month` → cyclical encoding (không quá quan trọng vì signal yếu, nhưng có thể tận dụng).

In [ ]:
def parse_loan_types(s):
    if pd.isna(s) or s in ('', 'Not Specified'):
        return []
    parts = re.split(r',\s*(?:and\s+)?', str(s))
    return [p.strip() for p in parts if p.strip() and p.strip() != 'Not Specified']

df['loan_list'] = df['Type_of_Loan'].apply(parse_loan_types)
df['Num_Loan_Types'] = df['loan_list'].apply(len)

# Top loan types phổ biến
all_types = [t for lst in df['loan_list'] for t in lst]
top_loans = pd.Series(all_types).value_counts().head(5).index.tolist()
print('Top 5 loan types:', top_loans)

for lt in top_loans:
    col_name = 'Has_' + lt.replace(' ', '_').replace('-', '_')
    df[col_name] = df['loan_list'].apply(lambda lst, t=lt: int(t in lst))

df = df.drop(columns=['loan_list', 'Type_of_Loan'])
print('Feature mới:', [c for c in df.columns if c.startswith('Has_')] + ['Num_Loan_Types'])

## 4. EDA (sau khi clean)

In [ ]:
# Phân phối target
plt.figure(figsize=(6, 4))
order = ['Poor', 'Standard', 'Good']
sns.countplot(x=df['Credit_Score'], order=order, palette=['#d9534f', '#f0ad4e', '#5cb85c'])
plt.title('Phân phối Credit_Score')
plt.show()

In [ ]:
# Boxplot các biến số quan trọng theo target
key_num = ['Delay_from_due_date', 'Credit_History_Months', 'Outstanding_Debt',
           'Monthly_Inhand_Salary', 'Interest_Rate', 'Num_of_Delayed_Payment']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, key_num):
    sns.boxplot(data=df, x='Credit_Score', y=col, order=order, ax=ax,
                palette=['#d9534f', '#f0ad4e', '#5cb85c'])
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Tỉ lệ target theo các biến phân loại quan trọng
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']):
    ct = pd.crosstab(df[col], df['Credit_Score'], normalize='index').loc[:, order]
    ct.plot(kind='bar', stacked=True, ax=ax,
            color=['#d9534f', '#f0ad4e', '#5cb85c'])
    ax.set_title(f'Credit_Score theo {col}')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='', loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## 5. Chuẩn bị dữ liệu train

**Split theo `Customer_ID`** để tránh cùng 1 khách hàng xuất hiện ở cả train và test (data leakage).

In [ ]:
# Cột không dùng làm feature
drop_cols = ['ID', 'Name', 'SSN', 'Month', 'Credit_History_Age']
groups = df['Customer_ID']
y_raw = df['Credit_Score']

X = df.drop(columns=drop_cols + ['Customer_ID', 'Credit_Score'])

# Encode target (giữ thứ tự Poor < Standard < Good)
target_order = ['Poor', 'Standard', 'Good']
y = y_raw.map({v: i for i, v in enumerate(target_order)})

print('X shape:', X.shape)
print('Feature columns:', X.columns.tolist())

In [ ]:
# Phân nhóm cột
ordinal_cols = ['Credit_Mix']
credit_mix_order = [['Bad', 'Standard', 'Good']]

onehot_cols = ['Occupation', 'Payment_of_Min_Amount', 'Payment_Behaviour']

numeric_cols = [c for c in X.columns if c not in ordinal_cols + onehot_cols]

print(f'Numeric ({len(numeric_cols)}):', numeric_cols)
print(f'Ordinal ({len(ordinal_cols)}):', ordinal_cols)
print(f'OneHot ({len(onehot_cols)}):', onehot_cols)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc', StandardScaler())
        ]), numeric_cols),
        ('ord', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('oe', OrdinalEncoder(categories=credit_mix_order,
                                   handle_unknown='use_encoded_value', unknown_value=-1))
        ]), ordinal_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), onehot_cols),
    ],
    remainder='drop'
)

In [ ]:
# Split theo Customer_ID
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train customers: {groups.iloc[train_idx].nunique()}, Test customers: {groups.iloc[test_idx].nunique()}')
print(f'Overlap customers: {len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))}  (phải = 0)')
print('\nTrain class distribution:')
print(y_train.value_counts(normalize=True).round(3).sort_index())
print('Test class distribution:')
print(y_test.value_counts(normalize=True).round(3).sort_index())

## 6. Train models

In [ ]:
def evaluate(pipe, X_te, y_te, name):
    pred = pipe.predict(X_te)
    proba = pipe.predict_proba(X_te)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average='macro')
    auc = roc_auc_score(y_te, proba, multi_class='ovr', average='macro')
    print(f'{name:20s} | Acc: {acc:.4f} | F1-macro: {f1:.4f} | AUC-ovr: {auc:.4f}')
    return {'model': name, 'accuracy': acc, 'f1_macro': f1, 'auc_ovr': auc,
            'pred': pred, 'proba': proba}

In [ ]:
results = []

# Baseline — Logistic Regression
pipe_lr = Pipeline([
    ('pre', preprocessor),
    ('model', LogisticRegression(max_iter=1000, multi_class='multinomial',
                                  random_state=RANDOM_STATE))
])
pipe_lr.fit(X_train, y_train)
results.append(evaluate(pipe_lr, X_test, y_test, 'LogisticRegression'))

In [ ]:
# LightGBM
pipe_lgbm = Pipeline([
    ('pre', preprocessor),
    ('model', LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        random_state=RANDOM_STATE, verbose=-1, class_weight='balanced'
    ))
])
pipe_lgbm.fit(X_train, y_train)
results.append(evaluate(pipe_lgbm, X_test, y_test, 'LightGBM'))

In [ ]:
# XGBoost
pipe_xgb = Pipeline([
    ('pre', preprocessor),
    ('model', XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=7,
        random_state=RANDOM_STATE, verbosity=0, eval_metric='mlogloss'
    ))
])
pipe_xgb.fit(X_train, y_train)
results.append(evaluate(pipe_xgb, X_test, y_test, 'XGBoost'))

In [ ]:
res_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('pred', 'proba')} for r in results])
res_df = res_df.sort_values('f1_macro', ascending=False).reset_index(drop=True)
res_df

In [ ]:
best_name = res_df.iloc[0]['model']
best_pipe = {'LogisticRegression': pipe_lr, 'LightGBM': pipe_lgbm, 'XGBoost': pipe_xgb}[best_name]
best_result = next(r for r in results if r['model'] == best_name)
print(f'Best model: {best_name}')

## 7. Đánh giá chi tiết

In [ ]:
print(classification_report(y_test, best_result['pred'], target_names=target_order))

In [ ]:
cm = confusion_matrix(y_test, best_result['pred'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_order, yticklabels=target_order)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title(f'Confusion Matrix — {best_name}')
plt.show()

## 8. Convert Probability → FICO Score (300–850)

Từ `predict_proba` (3 lớp), tính **expected ordinal**:
$$E[class] = 0 \cdot P(\text{Poor}) + 1 \cdot P(\text{Standard}) + 2 \cdot P(\text{Good}) \in [0, 2]$$
Rồi rescale về thang FICO:
$$score = 300 + \frac{E[class]}{2} \cdot 550$$

In [ ]:
def proba_to_fico(proba):
    """proba: (n_samples, 3) theo thứ tự [Poor, Standard, Good]."""
    expected = proba @ np.array([0, 1, 2])
    return 300 + expected / 2 * 550

def fico_to_rating(score):
    if score < 580: return 'Poor'
    if score < 670: return 'Fair'
    if score < 740: return 'Good'
    if score < 800: return 'Very Good'
    return 'Exceptional'

fico_scores = proba_to_fico(best_result['proba'])

plt.figure(figsize=(10, 4))
sns.histplot(fico_scores, bins=50, kde=True)
for x, lbl in [(580, 'Fair'), (670, 'Good'), (740, 'Very Good'), (800, 'Exceptional')]:
    plt.axvline(x, color='gray', linestyle='--', alpha=0.5)
    plt.text(x + 2, plt.ylim()[1] * 0.9, lbl, fontsize=9, color='gray')
plt.title(f'Phân phối FICO score (convert từ {best_name})')
plt.xlabel('FICO Score')
plt.show()

print(pd.Series([fico_to_rating(s) for s in fico_scores]).value_counts())

## 9. SHAP — giải thích mô hình

Chỉ áp dụng cho tree-based (LightGBM/XGBoost). Với multi-class, `shap_values` là list theo từng class.

In [ ]:
import shap

if best_name in ('LightGBM', 'XGBoost'):
    X_te_trans = best_pipe.named_steps['pre'].transform(X_test)
    fn = best_pipe.named_steps['pre'].get_feature_names_out()

    sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_te_trans), 500, replace=False)
    explainer = shap.TreeExplainer(best_pipe.named_steps['model'])
    shap_values = explainer.shap_values(X_te_trans[sample_idx])

    # Summary plot cho class 'Good' (index 2) — ảnh hưởng đến việc được xếp Good
    print("SHAP summary — feature nào đẩy prediction về 'Good':")
    if isinstance(shap_values, list):  # LGBM style
        shap.summary_plot(shap_values[2], X_te_trans[sample_idx],
                          feature_names=fn, max_display=15, show=True)
    else:  # XGB new API: (n, feat, class)
        shap.summary_plot(shap_values[..., 2], X_te_trans[sample_idx],
                          feature_names=fn, max_display=15, show=True)
else:
    print('Bỏ qua SHAP vì model không phải tree-based.')

## 10. Export artifacts

In [ ]:
# 1. Full pipeline
joblib.dump(best_pipe, OUT_DIR / 'model.pkl')

# 2. Schema — Streamlit dùng để dựng form
schema = {
    'task': 'classification_3class',
    'target': 'Credit_Score',
    'target_order': target_order,  # index 0=Poor, 1=Standard, 2=Good
    'best_model': best_name,
    'metrics': {k: float(v) for k, v in res_df.iloc[0].to_dict().items() if k != 'model'},
    'input_columns': list(X.columns),
    'numeric_cols': numeric_cols,
    'ordinal_cols': ordinal_cols,
    'onehot_cols': onehot_cols,
    'credit_mix_order': credit_mix_order[0],
    'categorical_values': {
        c: sorted([str(v) for v in df[c].dropna().unique().tolist()])
        for c in onehot_cols + ordinal_cols
    },
    'numeric_ranges': {
        c: {'min': float(df[c].min()), 'max': float(df[c].max()),
            'mean': float(df[c].mean()), 'median': float(df[c].median())}
        for c in numeric_cols if df[c].notna().any()
    },
    'fico_conversion': {
        'formula': 'score = 300 + (0*P_poor + 1*P_std + 2*P_good) / 2 * 550',
        'bins': [
            {'max': 580, 'label': 'Poor'},
            {'max': 670, 'label': 'Fair'},
            {'max': 740, 'label': 'Good'},
            {'max': 800, 'label': 'Very Good'},
            {'max': 850, 'label': 'Exceptional'}
        ]
    }
}

with open(OUT_DIR / 'features.json', 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)

print('Exported:')
for p in OUT_DIR.iterdir():
    print(f'  {p.name:30s} ({p.stat().st_size / 1024:.1f} KB)')

## 11. Sanity check — load lại và predict thử

In [ ]:
loaded = joblib.load(OUT_DIR / 'model.pkl')
sample = X_test.iloc[[0]]
proba = loaded.predict_proba(sample)[0]
pred_class = target_order[loaded.predict(sample)[0]]
fico = proba_to_fico(proba.reshape(1, -1))[0]
rating = fico_to_rating(fico)
actual_class = target_order[y_test.iloc[0]]

print('Input:')
print(sample.T)
print()
print(f'Actual class:  {actual_class}')
print(f'Predicted:     {pred_class}')
print(f'Probabilities: Poor={proba[0]:.3f} | Standard={proba[1]:.3f} | Good={proba[2]:.3f}')
print(f'FICO score:    {fico:.1f}  →  {rating}')

## 12. (Colab) Tải file về máy — nếu chưa dùng Drive sync

In [ ]:
# from google.colab import files
# for name in ['model.pkl', 'features.json']:
#     files.download(str(OUT_DIR / name))